# Story #10 — Đóng gói mô hình cuối cùng

Cấu hình tốt nhất đo được qua toàn bộ thí nghiệm Sprint 3 (đặc trưng khoảng cách + quét trọng số lớp): **XGBoost, có `seller_customer_distance_km`, `scale_pos_weight=6.0`** — F1-tối-đa-qua-ngưỡng 0.353, ngưỡng tối ưu ~0.539 (Task #58). Notebook này huấn luyện lại đúng cấu hình đó, lưu model xuống `models/xgboost_final.pkl` kèm ngưỡng quyết định vào `models/final_model.json` (API ở Story #11 cần biết ngưỡng này — 0.5 không phải mốc đúng), rồi **verify**: load lại model từ file, dự đoán trên cùng tập test, kết quả phải khớp 100% với lúc huấn luyện — đúng yêu cầu AC của Story #10.

## 1. Huấn luyện cấu hình cuối cùng, đo lại metric trên test

In [1]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import (
    precision_recall_curve, average_precision_score,
    f1_score, precision_score, recall_score,
)
from xgboost import XGBClassifier

train_df = pd.read_csv("../data/processed/orders_features_train.csv", low_memory=False)
test_df = pd.read_csv("../data/processed/orders_features_test.csv", low_memory=False)

bool_cols = ["payment_has_boleto", "payment_has_credit_card", "payment_has_debit_card",
             "payment_has_not_defined", "payment_has_voucher", "items_multi_seller"]
for df in (train_df, test_df):
    df["is_delayed"] = df["is_delayed"].astype(bool)
    for col in bool_cols:
        df[col] = df[col].astype("boolean")

X_train = train_df.drop(columns=["order_id", "is_delayed"])
y_train = train_df["is_delayed"].astype(int)
X_test = test_df.drop(columns=["order_id", "is_delayed"])
y_test = test_df["is_delayed"].astype(int)

SCALE_POS_WEIGHT = 6.0

model = XGBClassifier(scale_pos_weight=SCALE_POS_WEIGHT, random_state=42, n_jobs=-1, eval_metric="logloss")
model.fit(X_train, y_train)

y_score = model.predict_proba(X_test)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_test, y_score)
f1_per_threshold = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
best_idx = f1_per_threshold.argmax()
best_threshold = float(thresholds[best_idx])

y_pred = (y_score >= best_threshold).astype(int)

metrics = {
    "scale_pos_weight": SCALE_POS_WEIGHT,
    "decision_threshold": best_threshold,
    "f1": f1_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred),
    "recall": recall_score(y_test, y_pred),
    "average_precision_pr_auc": float(average_precision_score(y_test, y_score)),
    "n_features": X_train.shape[1],
}
print(json.dumps(metrics, indent=2))

{
  "scale_pos_weight": 6.0,
  "decision_threshold": 0.5390854477882385,
  "f1": 0.3529725406558251,
  "precision": 0.302836230558097,
  "recall": 0.4230031948881789,
  "average_precision_pr_auc": 0.29312445613559684,
  "n_features": 76
}


## 2. Lưu model + metadata (ngưỡng quyết định, danh sách đặc trưng — cần cho Story #11 API)

In [2]:
models_dir = Path("../models")

joblib.dump(model, models_dir / "xgboost_final.pkl")

final_model_info = {
    **metrics,
    "model_file": "xgboost_final.pkl",
    "feature_columns": list(X_train.columns),
}
with open(models_dir / "final_model.json", "w", encoding="utf-8") as f:
    json.dump(final_model_info, f, indent=2, ensure_ascii=False)

print("Da luu models/xgboost_final.pkl va models/final_model.json")

Da luu models/xgboost_final.pkl va models/final_model.json


## 3. Verify: load lại model từ file, dự đoán phải khớp 100% với lúc huấn luyện (AC của Story #10)

In [3]:
reloaded_model = joblib.load(models_dir / "xgboost_final.pkl")
y_score_reloaded = reloaded_model.predict_proba(X_test)[:, 1]

assert np.array_equal(y_score, y_score_reloaded), "Du doan sau khi load lai KHONG khop voi luc huan luyen!"
print("OK: du doan sau khi load lai model tu file khop 100% voi luc huan luyen.")
print(f"Model cuoi cung: XGBoost, scale_pos_weight={SCALE_POS_WEIGHT}, decision_threshold={best_threshold:.4f}")
print(f"F1={metrics['f1']:.4f}  Precision={metrics['precision']:.4f}  Recall={metrics['recall']:.4f}  "
      f"PR-AUC={metrics['average_precision_pr_auc']:.4f}")

OK: du doan sau khi load lai model tu file khop 100% voi luc huan luyen.
Model cuoi cung: XGBoost, scale_pos_weight=6.0, decision_threshold=0.5391
F1=0.3530  Precision=0.3028  Recall=0.4230  PR-AUC=0.2931
